
# Assignment 4 for Course 1MS041
Make sure you pass the `# ... Test` cells and
 submit your solution notebook in the corresponding assignment on the course website. You can submit multiple times before the deadline and your highest score will be used.

---
## Assignment 4, PROBLEM 1
Maximum Points = 24


    This time the assignment only consists of one problem, but we will do a more comprehensive analysis instead.

Consider the dataset `Corona_NLP_train.csv` that you can get from the course website [git](https://github.com/datascience-intro/1MS041-2024/blob/main/notebooks/data/Corona_NLP_train.csv). The data is "Coronavirus tweets NLP - Text Classification" that can be found on [kaggle](https://www.kaggle.com/datasets/datatattle/covid-19-nlp-text-classification). The data has several columns, but we will only be working with `OriginalTweet`and `Sentiment`.

1. [3p] Load the data and filter out those tweets that have `Sentiment`=`Neutral`. Let $X$ represent the `OriginalTweet` and let 
    $$
        Y = 
        \begin{cases}
        1 & \text{if sentiment is towards positive}
        \\
        0 & \text{if sentiment is towards negative}.
        \end{cases}
    $$
    Put the resulting arrays into the variables $X$ and $Y$. Split the data into three parts, train/test/validation where train is 60% of the data, test is 15% and validation is 25% of the data. Do not do this randomly, this is to make sure that we all did the same splits (we are in this case assuming the data is IID as presented in the dataset). That is [train,test,validation] is the splitting layout.

2. [4p] There are many ways to solve this classification problem. The first main issue to resolve is to convert the $X$ variable to something that you can feed into a machine learning model. For instance, you can first use [`CountVectorizer`](https://scikit-learn.org/1.5/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html) as the first step. The step that comes after should be a `LogisticRegression` model, but for this to work you need to put together the `CountVectorizer` and the `LogisticRegression` model into a [`Pipeline`](https://scikit-learn.org/1.5/modules/generated/sklearn.pipeline.Pipeline.html#sklearn.pipeline.Pipeline). Fill in the variable `model` such that it accepts the raw text as input and outputs a number $0$ or $1$, make sure that `model.predict_proba` works for this. **Hint: You might need to play with the parameters of LogisticRegression to get convergence, make sure that it doesn't take too long or the autograder might kill your code**
3. [3p] Use your trained model and calculate the precision and recall on both classes. Fill in the corresponding variables with the answer.
4. [3p] Let us now define a cost function
    * A positive tweet that is classified as negative will have a cost of 1
    * A negative tweet that is classified as positive will have a cost of 5
    * Correct classifications cost 0
    
    complete filling the function `cost` to compute the cost of a prediction model under a certain prediction threshold (recall our precision recall lecture and the `predict_proba` function from trained models). 

5. [4p] Now, we wish to select the threshold of our classifier that minimizes the cost, fill in the selected threshold value in value `optimal_threshold`.
6. [4p] With your newly computed threshold value, compute the cost of putting this model in production by computing the cost using the validation data. Also provide a confidence interval of the cost using Hoeffdings inequality with a 99% confidence.
7. [3p] Let $t$ be the threshold you found and $f$ the model you fitted (one of the outputs of `predict_proba`), if we define the random variable
    $$
        C = (1-1_{f(X)\geq t})Y+5(1-Y)1_{f(X) \geq t}
    $$
    then $C$ denotes the cost of a randomly chosen tweet. In the previous step we estimated $\mathbb{E}[C]$ using the empirical mean. However, since the threshold is chosen to minimize cost it is likely that $C=0$ or $C=1$ than $C=5$ as such it will have a low variance. Compute the empirical variance of $C$ on the validation set. What would be the confidence interval if we used Bennett's inequality instead of Hoeffding in point 6 but with the computed empirical variance as our guess for the variance?

In [ ]:
# Part 1

import numpy as np
import pandas as pd

# data file load korchi. Autograder-o ei same path use korbe.
df = pd.read_csv("data/Corona_NLP_train.csv", encoding="latin1")

# Neutral tweet bad dite hobe, karon question bole only positive vs negative classification.
df = df[df["Sentiment"] != "Neutral"].copy()

# X holo raw tweet text. CountVectorizer raw string input ney, tai X 1D numpy array hote hobe.
X = df["OriginalTweet"].astype(str).to_numpy()

# Y banacchi:
# Positive / Extremely Positive  -> 1
# Negative / Extremely Negative  -> 0
Y = df["Sentiment"].apply(lambda s: 1 if "Positive" in s else 0).to_numpy()

# Random split kora jabe na. Sequential split korte bolse.
n = len(X)
n_train = int(0.60 * n)
n_test = int(0.15 * n)

# Layout: [train, test, validation]
X_train = X[:n_train]
Y_train = Y[:n_train]

X_test = X[n_train:n_train + n_test]
Y_test = Y[n_train:n_train + n_test]

X_valid = X[n_train + n_test:]
Y_valid = Y[n_train + n_test:]

# Quick check: shape gula thik ase kina dekhar jonno
print(X_train.shape, X_test.shape, X_valid.shape)
print(Y_train.shape, Y_test.shape, Y_valid.shape)


In [ ]:
# Part 2

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression

# Pipeline use korchi karon model ke raw text input dite hobe.
# Step 1: CountVectorizer text ke word-count feature e convert kore.
# Step 2: LogisticRegression sei feature diye positive/negative classify kore.
model = Pipeline([
    ("vectorizer", CountVectorizer(lowercase=True, stop_words="english", min_df=2)),
    ("classifier", LogisticRegression(max_iter=1000, solver="liblinear"))
])

# Training data diye model train korchi.
model.fit(X_train, Y_train)


In [ ]:
# Part 3

from sklearn.metrics import precision_score, recall_score

# Test set e prediction nicchi.
Y_pred_test = model.predict(X_test)

# pos_label=0 dile class 0 er precision/recall pabo.
# pos_label=1 dile class 1 er precision/recall pabo.
precision_0 = precision_score(Y_test, Y_pred_test, pos_label=0, zero_division=0)
precision_1 = precision_score(Y_test, Y_pred_test, pos_label=1, zero_division=0)
recall_0 = recall_score(Y_test, Y_pred_test, pos_label=0, zero_division=0)
recall_1 = recall_score(Y_test, Y_pred_test, pos_label=1, zero_division=0)

print("precision_0 =", precision_0)
print("precision_1 =", precision_1)
print("recall_0 =", recall_0)
print("recall_1 =", recall_1)


In [ ]:
# Part 4

def cost(model, threshold, X, Y):
    import numpy as np
    
    # predict_proba(X) duita probability dey: [P(Y=0), P(Y=1)]
    # amader positive class holo Y=1, tai column 1 nibo.
    prob_positive = model.predict_proba(X)[:, 1]
    
    # Threshold rule:
    # probability >= threshold hole positive predict korbo, naile negative.
    prediction = (prob_positive >= threshold).astype(int)
    
    # Cost rule:
    # True positive tweet but negative predict => cost 1
    false_negative_cost = ((Y == 1) & (prediction == 0)) * 1
    
    # True negative tweet but positive predict => cost 5
    false_positive_cost = ((Y == 0) & (prediction == 1)) * 5
    
    # Average cost return korte hobe.
    total_cost = false_negative_cost + false_positive_cost
    return np.mean(total_cost)


In [ ]:
# Part 5

import numpy as np

# Test set er positive probability ber korchi.
probs_test = model.predict_proba(X_test)[:, 1]

# Threshold choose korar jonno candidate threshold banacchi.
# Unique probability gula important, karon prediction only oi points e change hoy.
candidate_thresholds = np.unique(probs_test)

# 0 and 1 include korchi jate boundary cases-o check hoy.
candidate_thresholds = np.concatenate(([0.0], candidate_thresholds, [1.0]))

# Duplicate remove and sorted.
candidate_thresholds = np.unique(candidate_thresholds)

# Prottek threshold er cost calculate korchi.
costs = np.array([cost(model, t, X_test, Y_test) for t in candidate_thresholds])

# Minimum cost jekhane hoy shei threshold select korchi.
best_index = np.argmin(costs)
optimal_threshold = float(candidate_thresholds[best_index])
cost_at_optimal_threshold = float(costs[best_index])

print("optimal_threshold =", optimal_threshold)
print("cost_at_optimal_threshold =", cost_at_optimal_threshold)


In [ ]:
# Part 6

import numpy as np

# Validation set e production cost estimate korchi.
cost_at_optimal_threshold_valid = float(cost(model, optimal_threshold, X_valid, Y_valid))

# Hoeffding confidence interval, confidence = 99%, alpha = 0.01
# Cost C always 0 theke 5 er moddhe, tai range = 5 - 0 = 5.
alpha = 0.01
n_valid = len(Y_valid)
cost_range = 5

# Hoeffding: epsilon = range * sqrt(log(2/alpha)/(2n))
epsilon_hoeffding = cost_range * np.sqrt(np.log(2 / alpha) / (2 * n_valid))

cost_interval_valid = (
    cost_at_optimal_threshold_valid - epsilon_hoeffding,
    cost_at_optimal_threshold_valid + epsilon_hoeffding
)

assert(type(cost_interval_valid) == tuple)
assert(len(cost_interval_valid) == 2)

print("validation cost =", cost_at_optimal_threshold_valid)
print("99% Hoeffding interval =", cost_interval_valid)


In [ ]:
# Part 7

import numpy as np
from scipy.optimize import brentq

# Validation set e probability ber korchi.
prob_valid = model.predict_proba(X_valid)[:, 1]

# Indicator: f(X) >= t hole 1, otherwise 0
indicator_positive = (prob_valid >= optimal_threshold).astype(int)

# Question er formula:
# C = (1 - 1_{f(X)>=t})Y + 5(1-Y)1_{f(X)>=t}
# Ei C each validation point er cost.
C = (1 - indicator_positive) * Y_valid + 5 * (1 - Y_valid) * indicator_positive

# Empirical variance of C
variance_of_C = float(np.var(C, ddof=0))

# Bennett inequality diye confidence interval banabo.
# Bound: P(|mean-E| >= eps) <= 2 exp(-(n*v/b^2) h(b*eps/v))
# where h(u)=(1+u)log(1+u)-u, b=5, v=variance guess.
mean_C = float(np.mean(C))
alpha = 0.01
n_valid = len(C)
b = 5.0
v = variance_of_C

def h(u):
    return (1 + u) * np.log(1 + u) - u

# v zero hole variance nai, interval mean er kachakachi thake.
if v == 0:
    epsilon_bennett = 0.0
else:
    def bennett_equation(eps):
        return 2 * np.exp(-(n_valid * v / (b ** 2)) * h((b * eps) / v)) - alpha
    
    # epsilon 0 theke 5 er moddhe search korchi, karon cost range 0..5.
    epsilon_bennett = brentq(bennett_equation, 0, 5)

interval_of_C = (
    mean_C - epsilon_bennett,
    mean_C + epsilon_bennett
)

assert(type(interval_of_C) == tuple)
assert(len(interval_of_C) == 2)

print("variance_of_C =", variance_of_C)
print("99% Bennett interval =", interval_of_C)
